# BÀI TẬP VỀ NHÀ TUẦN 7: PHÂN LOẠI CẢM XÚC VĂN BẢN (SENTIMENT ANALYSIS)
## Xây dựng và So sánh Mô hình RNN, LSTM với Mạng Neural Cơ Bản và Các Phương Pháp Học Máy Truyền Thống

---
### 📌 THÔNG TIN BÀI TẬP
- **Học phần**: Xử lý Ngôn ngữ Tự nhiên (Natural Language Processing - NLP)
- **Chủ đề**: Mạng nơ-ron hồi quy (Recurrent Neural Networks - RNN) & Long Short-Term Memory (LSTM)
- **Bộ dữ liệu**: **IMDB Large Movie Review Dataset** (50.000 đánh giá phim: 25.000 Train, 25.000 Test; Cân bằng 50% Positive / 50% Negative)
- **Nhiệm vụ**: Phân loại đánh giá phim thành 2 lớp:
  - **1 (Positive - Tích cực)**: Người xem thích và đánh giá cao bộ phim.
  - **0 (Negative - Tiêu cực)**: Người xem chê bai, không thích bộ phim.

---
### 🎯 CÁC MÔ HÌNH THỰC HIỆN VÀ SO SÁNH:
1. **Phương pháp Học máy truyền thống (Traditional Machine Learning)**:
   - **Logistic Regression**: Dự đoán xác suất nhãn với vector đặc trưng TF-IDF.
   - **Support Vector Machine (LinearSVC)**: Phân loại dựa trên siêu phẳng tối đa hóa biên phân chia (Maximum Margin) với TF-IDF.
2. **Mạng Nơ-ron Cơ bản (Basic Neural Network)**:
   - **MLP (Multilayer Perceptron)**: Kết hợp Word Embedding + Global Average Pooling + Fully Connected Layers.
3. **Mạng Nơ-ron Hồi quy (Recurrent Architectures)**:
   - **Simple RNN**: Xử lý chuỗi từ tuần tự qua các hidden state thời gian.
   - **LSTM (Long Short-Term Memory)**: Mạng hồi quy có các cổng (Forget, Input, Output) và Cell State dài hạn.
   - **GRU (Gated Recurrent Unit)**: Kiến trúc rút gọn của LSTM với Reset và Update gate (mô hình mở rộng).


## 1. Cài đặt Môi trường và Import Thư viện
Thiết lập Seed ngẫu nhiên để đảm bảo khả năng tái lập kết quả (reproducibility) và kiểm tra thiết bị tính toán (GPU / CUDA hoặc CPU).


In [ ]:
import os
import re
import time
import string
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Cấu hình giao diện biểu đồ
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Kiểm tra thiết bị tính toán
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Thiết bị sử dụng: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"[*] GPU Name: {torch.cuda.get_device_name(0)}")


## 2. Tải và Khám phá Dữ liệu (Exploratory Data Analysis - EDA)
Bộ dữ liệu IMDB gồm 50.000 mẫu đánh giá phim. Chúng ta nạp tập huấn luyện (25.000 mẫu) và tập kiểm thử (25.000 mẫu).


In [ ]:
data_dir = os.path.join(".", "data")

train_file = os.path.join(data_dir, "train.parquet")
test_file = os.path.join(data_dir, "test.parquet")

df_train = pd.read_parquet(train_file)
df_test = pd.read_parquet(test_file)

print(f"[*] Kích thước tập Train: {df_train.shape}")
print(f"[*] Kích thước tập Test:  {df_test.shape}")

print("\nPhân bố nhãn tập Train:")
print(df_train['label'].value_counts())

print("\nPhân bố nhãn tập Test:")
print(df_test['label'].value_counts())


Xem trước 2 mẫu đánh giá (1 tích cực và 1 tiêu cực) để nắm được cấu trúc văn bản gốc:


In [ ]:
pos_sample = df_train[df_train['label'] == 1].iloc[0]['text']
neg_sample = df_train[df_train['label'] == 0].iloc[0]['text']

print("--- [MẪU REVIEW TÍCH CỰC (Label: 1)] ---")
print(pos_sample[:400] + "...")
print("\n--- [MẪU REVIEW TIÊU CỰC (Label: 0)] ---")
print(neg_sample[:400] + "...")


### Thống kê và Trực quan hóa Phân bố Độ dài Văn bản


In [ ]:
df_train['word_count'] = df_train['text'].apply(lambda t: len(t.split()))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df_train['word_count'], bins=50, kde=True, color='teal')
plt.title("Phân bố số lượng từ trong mỗi Review (Train)")
plt.xlabel("Số từ")
plt.ylabel("Tần suất")
plt.axvline(df_train['word_count'].median(), color='red', linestyle='--', label=f"Median: {df_train['word_count'].median():.0f}")
plt.legend()

plt.subplot(1, 2, 2)
sns.boxplot(x='label', y='word_count', data=df_train, palette=['#e74c3c', '#2ecc71'])
plt.xticks([0, 1], ['Negative (0)', 'Positive (1)'])
plt.title("So sánh độ dài giữa Review Tích cực & Tiêu cực")
plt.xlabel("Cảm xúc")
plt.ylabel("Số từ")

plt.tight_layout()
plt.show()

print(f"Độ dài trung bình: {df_train['word_count'].mean():.1f} từ")
print(f"Độ dài trung vị:   {df_train['word_count'].median():.1f} từ")
print(f"Review dài nhất:   {df_train['word_count'].max()} từ")
print(f"Review ngắn nhất:  {df_train['word_count'].min()} từ")


## 3. Tiền xử lý Văn bản (Text Preprocessing)
Các bước chuẩn hóa:
1. Loại bỏ các thẻ HTML (như `<br />`, `<p>`).
2. Loại bỏ đường dẫn URL.
3. Chuyển đổi toàn bộ thành chữ thường (lowercasing).
4. Loại bỏ các ký tự dấu câu đặc biệt.
5. Xóa bỏ khoảng trắng thừa.


In [ ]:
def clean_text(text: str) -> str:
    # 1. Xóa thẻ HTML
    text = re.sub(r"<br\s*/?>", " ", text)
    # 2. Xóa URL
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    # 3. Lowercase
    text = text.lower()
    # 4. Loại bỏ dấu câu
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    # 5. Xóa khoảng trắng thừa
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("[*] Đang làm sạch dữ liệu văn bản...")
df_train['clean_text'] = df_train['text'].apply(clean_text)
df_test['clean_text'] = df_test['text'].apply(clean_text)

print("[*] Ví dụ sau khi tiền xử lý:")
print(df_train['clean_text'].iloc[0][:300] + "...")


## 4. Các Phương pháp Học máy Truyền thống (Baseline Models)
### 4.1 Trích xuất Đặc trưng với TF-IDF (Term Frequency - Inverse Document Frequency)
$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$
Trong đó:
- $\text{TF}(t, d)$: Tần suất xuất hiện của từ $t$ trong văn bản $d$.
- $\text{IDF}(t, D) = \log\left(\frac{1 + |D|}{1 + |\{d \in D : t \in d\}|}\right) + 1$: Nghịch đảo tần suất văn bản, phạt các từ xuất hiện quá phổ biến ở mọi nơi.

Chúng ta sử dụng kết hợp **unigram và bigram** (cụm 1 từ và 2 từ) với `max_features=25000`.


In [ ]:
print("[*] Trích xuất đặc trưng TF-IDF...")
t0 = time.time()
tfidf = TfidfVectorizer(max_features=25000, ngram_range=(1, 2), sublinear_tf=True)

X_train_tfidf = tfidf.fit_transform(df_train['clean_text'])
X_test_tfidf = tfidf.transform(df_test['clean_text'])

y_train = df_train['label'].values
y_test = df_test['label'].values

print(f"[*] Kích thước ma trận TF-IDF Train: {X_train_tfidf.shape}")
print(f"[*] Kích thước ma trận TF-IDF Test:  {X_test_tfidf.shape}")
print(f"[*] Thời gian trích xuất: {time.time() - t0:.2f}s")


### 4.2 Mô hình 1: Logistic Regression
Logistic Regression mô hình hóa xác suất một đánh giá là Positive ($y=1$) thông qua hàm Sigmoid:
$$P(y=1|x) = \sigma(w^T x + b) = \frac{1}{1 + e^{-(w^T x + b)}}$$


In [ ]:
print("---> Huấn luyện Logistic Regression...")
t0 = time.time()
lr_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)
lr_time = time.time() - t0

y_pred_lr = lr_model.predict(X_test_tfidf)
y_prob_lr = lr_model.predict_proba(X_test_tfidf)[:, 1]

lr_metrics = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, y_pred_lr),
    "Precision": precision_score(y_test, y_pred_lr),
    "Recall": recall_score(y_test, y_pred_lr),
    "F1-Score": f1_score(y_test, y_pred_lr),
    "ROC-AUC": roc_auc_score(y_test, y_prob_lr),
    "Train Time (s)": lr_time
}

print(f"[*] Accuracy: {lr_metrics['Accuracy']*100:.2f}% | F1: {lr_metrics['F1-Score']:.4f} | Time: {lr_time:.2f}s")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Negative', 'Positive']))


### 4.3 Mô hình 2: Support Vector Machine (LinearSVC)
Linear Support Vector Machine tìm siêu phẳng phân chia $w^T x + b = 0$ tối đa hóa lề (margin):
$$\min_{w, b} \frac{1}{2} \|w\|^2 + C \sum_{i=1}^N \max(0, 1 - y_i(w^T x_i + b))$$


In [ ]:
print("---> Huấn luyện Support Vector Machine (LinearSVC)...")
t0 = time.time()
svm_model = LinearSVC(C=0.5, max_iter=2000, random_state=42)
svm_model.fit(X_train_tfidf, y_train)
svm_time = time.time() - t0

y_pred_svm = svm_model.predict(X_test_tfidf)
y_score_svm = svm_model.decision_function(X_test_tfidf)

svm_metrics = {
    "Model": "SVM (LinearSVC)",
    "Accuracy": accuracy_score(y_test, y_pred_svm),
    "Precision": precision_score(y_test, y_pred_svm),
    "Recall": recall_score(y_test, y_pred_svm),
    "F1-Score": f1_score(y_test, y_pred_svm),
    "ROC-AUC": roc_auc_score(y_test, y_score_svm),
    "Train Time (s)": svm_time
}

print(f"[*] Accuracy: {svm_metrics['Accuracy']*100:.2f}% | F1: {svm_metrics['F1-Score']:.4f} | Time: {svm_time:.2f}s")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm, target_names=['Negative', 'Positive']))


## 5. Chuẩn bị Dữ liệu cho Mạng Nơ-ron (PyTorch)
Để đưa văn bản vào mạng nơ-ron:
1. Xây dựng **Vocabulary** (Từ điển) từ tập huấn luyện. Các từ ít xuất hiện hoặc không có trong từ điển được gán token `<UNK>` (index 1). Ký tự đệm là `<PAD>` (index 0).
2. Mã hóa văn bản thành chuỗi số nguyên (Sequence IDs) và cố định độ dài chuỗi bằng Padding/Truncating về `max_len = 256`.
3. Đóng gói vào PyTorch `Dataset` và `DataLoader`.


In [ ]:
class Vocabulary:
    def __init__(self, max_size=25000, min_freq=2):
        self.max_size = max_size
        self.min_freq = min_freq
        self.pad_token = "<PAD>"
        self.unk_token = "<UNK>"
        self.pad_idx = 0
        self.unk_idx = 1
        
        self.word2idx = {self.pad_token: self.pad_idx, self.unk_token: self.unk_idx}
        self.idx2word = {self.pad_idx: self.pad_token, self.unk_idx: self.unk_token}
        
    def build_vocab(self, texts):
        counter = Counter()
        for text in texts:
            counter.update(text.split())
            
        most_common = counter.most_common(self.max_size - 2)
        idx = 2
        for word, count in most_common:
            if count >= self.min_freq:
                self.word2idx[word] = idx
                self.idx2word[idx] = word
                idx += 1
                
    def encode(self, text, max_len=256):
        tokens = text.split()
        indices = [self.word2idx.get(w, self.unk_idx) for w in tokens[:max_len]]
        if len(indices) < max_len:
            indices += [self.pad_idx] * (max_len - len(indices))
        return indices

vocab = Vocabulary(max_size=25000, min_freq=2)
vocab.build_vocab(df_train['clean_text'])
print(f"[*] Kích thước từ điển (Vocabulary Size): {len(vocab.word2idx)}")

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, vocab: Vocabulary, max_len=256):
        self.labels = labels
        self.encoded_texts = [vocab.encode(t, max_len=max_len) for t in texts]
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return (
            torch.tensor(self.encoded_texts[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

max_len = 256
batch_size = 64

train_dataset = IMDBDataset(df_train['clean_text'], y_train, vocab, max_len=max_len)
test_dataset = IMDBDataset(df_test['clean_text'], y_test, vocab, max_len=max_len)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=(DEVICE.type=="cuda"))
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=(DEVICE.type=="cuda"))

print(f"[*] Số batch tập Train: {len(train_loader)} (mỗi batch {batch_size} mẫu)")
print(f"[*] Số batch tập Test:  {len(test_loader)}")


## 6. Xây dựng Các Kiến trúc Mạng Nơ-ron

### 6.1 Mạng Nơ-ron Cơ bản: Basic Feedforward Neural Network (MLP)
Mô hình cơ bản biểu diễn văn bản bằng cách:
1. Chiếu từng từ vào không gian nhúng (Word Embedding $d=128$).
2. Tính trung bình các vector nhúng (Global Average Pooling) bỏ qua padding.
3. Đưa qua lớp Fully Connected với hàm kích hoạt ReLU và Dropout để phân loại.

$$\text{Text} \xrightarrow{\text{Embedding}} [x_1, x_2, ..., x_T] \xrightarrow{\text{AvgPool}} \bar{x} \xrightarrow{\text{Dense + ReLU}} h \xrightarrow{\text{Dense}} \hat{y}$$


In [ ]:
class BasicMLP(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        # x: [batch_size, seq_len]
        embeds = self.embedding(x)  # [batch_size, seq_len, embed_dim]
        mask = (x != 0).unsqueeze(-1).float()  # loại bỏ pad tokens
        pooled = (embeds * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        out = self.fc1(pooled)
        out = self.relu(out)
        out = self.dropout(out)
        logits = self.fc2(out).squeeze(-1)
        return logits


### 6.2 Mạng Nơ-ron Hồi quy: Simple RNN (Recurrent Neural Network)
RNN xử lý văn bản theo từng bước thời gian $t$. Ở mỗi bước:
$$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$$
Nhược điểm lớn nhất: Khi lan truyền ngược qua thời gian (BPTT), đạo hàm liên tục nhân ma trận trọng số $W_{hh}^T$, dẫn đến hiện tượng **Triệt tiêu đạo hàm (Vanishing Gradient)** khi chuỗi dài ($>20-30$ từ).


In [ ]:
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_layers=1, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            nonlinearity='tanh'
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)  # bidirectional -> * 2
        
    def forward(self, x):
        embeds = self.embedding(x)
        out, h_n = self.rnn(embeds)
        # h_n shape: [num_layers * 2, batch_size, hidden_dim]
        last_hidden = torch.cat((h_n[-2, :, :], h_n[-1, :, :]), dim=1)
        last_hidden = self.dropout(last_hidden)
        logits = self.fc(last_hidden).squeeze(-1)
        return logits


### 6.3 Mạng Long Short-Term Memory (LSTM)
LSTM giải quyết triệt để vấn đề vanishing gradient nhờ **Cell State ($C_t$)** đóng vai trò như đường cao tốc thông tin, điều khiển bởi 3 cổng (Gates):
1. **Forget Gate**: Quyết định thông tin cũ nào bị lãng quên:
   $$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$$
2. **Input Gate & Candidate State**: Quyết định thông tin mới nào được lưu:
   $$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i), \quad \tilde{C}_t = \tanh(W_c [h_{t-1}, x_t] + b_c)$$
3. **Cập nhật Cell State**:
   $$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$
4. **Output Gate & Hidden State**:
   $$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o), \quad h_t = o_t \odot \tanh(C_t)$$


In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        
    def forward(self, x):
        embeds = self.embedding(x)
        out, (h_n, c_n) = self.lstm(embeds)
        last_hidden = torch.cat((h_n[-2, :, :], h_n[-1, :, :]), dim=1)
        last_hidden = self.dropout(last_hidden)
        logits = self.fc(last_hidden).squeeze(-1)
        return logits


### 6.4 Mạng Gated Recurrent Unit (GRU) - Mô hình Mở rộng
GRU gộp Forget gate và Input gate thành Update gate ($z_t$), kết hợp Cell state và Hidden state, giúp huấn luyện nhanh hơn và ít tham số hơn LSTM.


In [ ]:
class GRUModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        
    def forward(self, x):
        embeds = self.embedding(x)
        out, h_n = self.gru(embeds)
        last_hidden = torch.cat((h_n[-2, :, :], h_n[-1, :, :]), dim=1)
        last_hidden = self.dropout(last_hidden)
        logits = self.fc(last_hidden).squeeze(-1)
        return logits


## 7. Quy trình Huấn luyện và Đánh giá (Training Loop)


In [ ]:
def train_epoch(model, dataloader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        total_loss += loss.item() * batch_x.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == batch_y).sum().item()
        total += batch_x.size(0)
        
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate_model(model, dataloader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_probs, all_labels = [], [], []
    
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        total_loss += loss.item() * batch_x.size(0)
        
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(batch_y.cpu().numpy().astype(int))
        total += batch_x.size(0)
        
    metrics = {
        "loss": total_loss / total,
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds),
        "recall": recall_score(all_labels, all_preds),
        "f1": f1_score(all_labels, all_preds),
        "roc_auc": roc_auc_score(all_labels, all_probs),
        "confusion_matrix": confusion_matrix(all_labels, all_preds),
        "preds": all_preds,
        "probs": all_probs,
        "labels": all_labels
    }
    return metrics

def run_training_pipeline(model_name, model, epochs=5, lr=1e-3):
    print(f"\n{'='*20} Huấn luyện {model_name} {'='*20}")
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    
    start_time = time.time()
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
        val_m = evaluate_model(model, test_loader, criterion)
        
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(val_m["loss"])
        history["val_acc"].append(val_m["accuracy"])
        
        print(f"Epoch {epoch:02d}/{epochs:02d} | Train Loss: {tr_loss:.4f} - Acc: {tr_acc*100:.2f}% | Val Loss: {val_m['loss']:.4f} - Val Acc: {val_m['accuracy']*100:.2f}%")
        
    elapsed = time.time() - start_time
    final_m = evaluate_model(model, test_loader, criterion)
    final_m["train_time"] = elapsed
    final_m["history"] = history
    final_m["num_params"] = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[*] Hoàn thành {model_name} sau {elapsed:.1f}s | Test Accuracy: {final_m['accuracy']*100:.2f}% | F1: {final_m['f1']:.4f}")
    return final_m


## 8. Tiến hành Huấn luyện 4 Mô hình Deep Learning


In [ ]:
vocab_size = len(vocab.word2idx)
epochs = 5

all_results = {}
all_results["Logistic Regression"] = lr_metrics
all_results["SVM (LinearSVC)"] = svm_metrics

# 1. Basic MLP
mlp_model = BasicMLP(vocab_size=vocab_size, embed_dim=128, hidden_dim=128, dropout=0.3).to(DEVICE)
all_results["Basic Neural Network (MLP)"] = run_training_pipeline("Basic MLP", mlp_model, epochs=epochs, lr=1e-3)

# 2. Simple RNN
rnn_model = RNNModel(vocab_size=vocab_size, embed_dim=128, hidden_dim=128, num_layers=1, dropout=0.3).to(DEVICE)
all_results["Simple RNN"] = run_training_pipeline("Simple RNN", rnn_model, epochs=epochs, lr=5e-4)

# 3. LSTM
lstm_model = LSTMModel(vocab_size=vocab_size, embed_dim=128, hidden_dim=128, num_layers=2, dropout=0.3).to(DEVICE)
all_results["LSTM"] = run_training_pipeline("LSTM", lstm_model, epochs=epochs, lr=1e-3)

# 4. GRU
gru_model = GRUModel(vocab_size=vocab_size, embed_dim=128, hidden_dim=128, num_layers=2, dropout=0.3).to(DEVICE)
all_results["GRU"] = run_training_pipeline("GRU", gru_model, epochs=epochs, lr=1e-3)


## 9. Bảng So sánh Tổng hợp và Trực quan hóa Kết quả
Tổng hợp tất cả các độ đo đánh giá chính:
- **Accuracy**: Tỷ lệ dự đoán đúng tổng thể.
- **Precision**: Độ chính xác của các dự đoán Positive.
- **Recall**: Khả năng bao quát các mẫu Positive thực tế.
- **F1-Score**: Trung bình điều hòa giữa Precision và Recall.
- **ROC-AUC**: Diện tích dưới đường cong ROC (khả năng phân tách hai lớp).
- **Train Time**: Thời gian huấn luyện (giây).
- **Parameters**: Số lượng tham số cần tối ưu.


In [ ]:
summary_rows = []
for model_name, m in all_results.items():
    summary_rows.append({
        "Model": model_name,
        "Accuracy (%)": round(m["Accuracy"] * 100 if "Accuracy" in m else m["accuracy"] * 100, 2),
        "Precision": round(m["Precision"] if "Precision" in m else m["precision"], 4),
        "Recall": round(m["Recall"] if "Recall" in m else m["recall"], 4),
        "F1-Score": round(m["F1-Score"] if "F1-Score" in m else m["f1"], 4),
        "ROC-AUC": round(m["ROC-AUC"] if "ROC-AUC" in m else m["roc_auc"], 4),
        "Train Time (s)": round(m["Train Time (s)"] if "Train Time (s)" in m else m["train_time"], 2),
        "Parameters": m.get("num_params", 25001)
    })

df_summary = pd.DataFrame(summary_rows)
display(df_summary)


### 9.1 Biểu đồ Cột So sánh Accuracy và F1-Score


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Biểu đồ Accuracy
sns.barplot(data=df_summary, x="Accuracy (%)", y="Model", palette="crest", ax=axes[0])
axes[0].set_title("So sánh Accuracy (%) giữa các Mô hình", fontsize=14, fontweight="bold")
axes[0].set_xlim(70, 95)
for i, v in enumerate(df_summary["Accuracy (%)"]):
    axes[0].text(v + 0.3, i, f"{v:.2f}%", va="center", fontweight="bold")

# Biểu đồ F1-Score
sns.barplot(data=df_summary, x="F1-Score", y="Model", palette="viridis", ax=axes[1])
axes[1].set_title("So sánh F1-Score giữa các Mô hình", fontsize=14, fontweight="bold")
axes[1].set_xlim(0.7, 0.95)
for i, v in enumerate(df_summary["F1-Score"]):
    axes[1].text(v + 0.003, i, f"{v:.4f}", va="center", fontweight="bold")

plt.tight_layout()
plt.show()


### 9.2 Biểu đồ Đường (Learning Curves) Quá trình Huấn luyện


In [ ]:
plt.figure(figsize=(15, 5))
dl_names = ["Basic Neural Network (MLP)", "Simple RNN", "LSTM", "GRU"]

plt.subplot(1, 2, 1)
for name in dl_names:
    if name in all_results and "history" in all_results[name]:
        plt.plot(range(1, epochs + 1), all_results[name]["history"]["val_loss"], marker="o", label=name)
plt.title("Validation Loss qua các Epochs", fontsize=13, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)

plt.subplot(1, 2, 2)
for name in dl_names:
    if name in all_results and "history" in all_results[name]:
        accs = [a * 100 for a in all_results[name]["history"]["val_acc"]]
        plt.plot(range(1, epochs + 1), accs, marker="s", label=name)
plt.title("Validation Accuracy (%) qua các Epochs", fontsize=13, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()


### 9.3 Ma trận Nhầm lẫn (Confusion Matrix)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

models_list = list(all_results.keys())
for idx, name in enumerate(models_list):
    m = all_results[name]
    cm = m.get("confusion_matrix")
    if cm is None:
        if "y_pred_lr" in locals() and name == "Logistic Regression":
            cm = confusion_matrix(y_test, y_pred_lr)
        elif "y_pred_svm" in locals() and name == "SVM (LinearSVC)":
            cm = confusion_matrix(y_test, y_pred_svm)
            
    acc = m.get("Accuracy", m.get("accuracy", 0)) * 100
    f1 = m.get("F1-Score", m.get("f1", 0))
    
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[idx],
                xticklabels=["Negative (0)", "Positive (1)"],
                yticklabels=["Negative (0)", "Positive (1)"])
    axes[idx].set_title(f"{name}\nAccuracy: {acc:.2f}% | F1: {f1:.4f}", fontsize=11, fontweight="bold")
    axes[idx].set_xlabel("Predicted Label")
    axes[idx].set_ylabel("True Label")

plt.tight_layout()
plt.show()


## 10. Phân tích Chi tiết và Thảo luận Kết quả (Discussion)

Dựa trên kết quả thực nghiệm 50.000 đánh giá IMDB, ta rút ra các nhận định then chốt:

### 10.1 So sánh RNN vs LSTM:
- **Hiện tượng Triệt tiêu Đạo hàm (Vanishing Gradient)**: Simple RNN chỉ cập nhật trạng thái ẩn qua một ma trận chuyển đổi lặp lại và hàm $\tanh$. Khi văn bản dài hàng trăm từ (độ dài trung bình IMDB là ~230 từ), thông tin từ những từ đầu tiên bị mất dần, dẫn đến độ chính xác của Simple RNN thường chỉ đạt quanh mức **80 - 83%**, khó học được các ngữ cảnh xa.
- **Ưu thế vượt trội của LSTM**: Nhờ cấu trúc **Cell State ($C_t$)** hoạt động như một băng chuyền thông tin tuyến tính:
  $$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$
  Khi lan truyền ngược, đạo hàm $\frac{\partial C_t}{\partial C_{t-1}} = f_t$. Nếu cổng quên $f_t \approx 1$, đạo hàm được bảo toàn nguyên vẹn qua hàng trăm bước thời gian mà không bị nhân số mũ gây triệt tiêu. Nhờ vậy, LSTM đạt độ chính xác cao vượt trội (**~87 - 89%**).

### 10.2 So sánh Deep Learning với Machine Learning Truyền thống (Logistic Regression & SVM):
- **Hiệu quả ấn tượng của TF-IDF + Logistic Regression / LinearSVC**:
  + Cả hai mô hình truyền thống đều đạt độ chính xác rất cao (**~89 - 90%**), ngang ngửa hoặc thậm chí nhỉnh hơn một số mô hình deep learning cơ bản.
  + **Thời gian huấn luyện**: Cực kỳ nhanh (chỉ mất ~2 - 5 giây), trong khi các mô hình hồi quy cần nhiều phút trên GPU/CPU.
  + **Lý do**: Bài toán phân loại cảm xúc phim phụ thuộc rất nhiều vào các từ khóa cảm xúc đặc trưng (sentiment lexicon) như *"masterpiece", "brilliant", "terrible", "boring", "waste of time"*. Khi TF-IDF trích xuất cả unigram và bigram với 25.000 đặc trưng, mô hình tuyến tính đã nắm bắt được phần lớn tín hiệu cảm xúc quyết định.
- **Khi nào LSTM/GRU vượt trội hơn TF-IDF?**:
  + Khi câu có ngữ pháp phức tạp, cấu trúc đảo ngữ hoặc phủ định tinh tế:
    - Ví dụ: *"I thought this would be another boring typical Hollywood movie, but actually it was surprisingly great."* -> TF-IDF dễ bị nhiễu bởi từ *"boring"*, trong khi LSTM có thể duy trì trạng thái ngữ cảnh chuyển biến từ đầu câu đến cuối câu.
    - Cụm từ phủ định kép: *"not bad at all"*, *"hardly boring"*.

### 10.3 So sánh Mạng Neural Cơ bản (MLP) với RNN/LSTM:
- Basic MLP sử dụng Global Average Pooling trên các vector nhúng (Bag of Embeddings). Mô hình này độc lập với thứ tự từ và chỉ tổng hợp ngữ nghĩa chung, đạt khoảng **86 - 87%**.
- Mô hình LSTM và GRU duy trì trật tự tuần tự và trạng thái ẩn động, giúp biểu diễn sắc thái cảm xúc trọn vẹn hơn.


## 11. Thử nghiệm Dự đoán trên Đánh giá Tùy ý (Interactive Inference)
Dưới đây là hàm dự đoán cho phép nhập bất kỳ câu review nào bằng tiếng Anh để kiểm tra khả năng phân loại của mô hình LSTM:


In [ ]:
def predict_sentiment(text: str, model=lstm_model, vocab=vocab, max_len=256):
    model.eval()
    cleaned = clean_text(text)
    encoded = vocab.encode(cleaned, max_len=max_len)
    tensor_input = torch.tensor([encoded], dtype=torch.long).to(DEVICE)
    
    with torch.no_grad():
        logits = model(tensor_input)
        prob = torch.sigmoid(logits).item()
        
    label = "POSITIVE (Tích cực)" if prob >= 0.5 else "NEGATIVE (Tiêu cực)"
    color = "\033[92m" if prob >= 0.5 else "\033[91m"
    reset = "\033[0m"
    
    print(f"\nReview: \"{text}\"")
    print(f"-> Dự đoán: {color}{label}{reset} (Xác suất Positive: {prob*100:.2f}%)")
    return label, prob

# Thử nghiệm một số câu thực tế
test_samples = [
    "This movie was an absolute masterpiece! The acting was superb and the story kept me hooked from start to finish.",
    "Waste of time and money. Terrible script, wooden acting, and utterly boring.",
    "I was skeptical at first, but it turned out to be one of the best films I have seen this year. Not bad at all!",
    "The plot makes no sense whatsoever and the characters are thoroughly annoying."
]

print("[*] THỬ NGHIỆM DỰ ĐOÁN VỚI MÔ HÌNH LSTM:")
for sample in test_samples:
    predict_sentiment(sample)
